# Video huấn luyện Ultralight từ nhân vật AI bằng InfiniteTalk (ADR-060 mục 1)

Notebook chạy **trên Kaggle**, không chạy trên laptop. Nó bật ComfyUI không giao diện web với ComfyUI-WanVideoWrapper, rồi dùng
InfiniteTalk (dựa trên Wan2.1 I2V 14B) tạo video nhân vật nói theo tiếng VieNeu, 25 khung/giây. Video này **chỉ là dữ liệu huấn
luyện** cho Ultralight, không bao giờ phát live (ràng buộc 1 của CLAUDE.md: live luôn sinh hình và tiếng theo thời gian thực).

**Quy tắc:**
- Ảnh đầu vào là khung hình của nhân vật AI hiện tại (ADR-036), không đưa ảnh người thật vào.
- Mỗi nhân vật chỉ dùng cho một khách, một shop.
- Video, khung hình và `run.json` **không đưa vào git**. Tải về laptop, lưu ở `assets/avatars/<CHARACTER_ID>/training/`.

**Cách chạy:**
1. Tạo dataset riêng tư trên Kaggle chứa `start.png` (khung hình nhân vật, 480×832) và một file `.wav` (tiếng VieNeu). Bản thử
   ngắn: khoảng 13 giây. Bản huấn luyện: 3–5 phút, 20 giây đầu im lặng (ADR-060).
2. kaggle.com → Create → New Notebook → File → Import Notebook → chọn file này. Add Input → chọn dataset ở bước 1. Bản
   lưu và chạy (Save Version) **phải** gắn dataset sẵn; kagglehub chỉ tải được dataset khi chạy tương tác.
3. Session options: Accelerator = **GPU T4 x2**, Internet = **On**.
4. Sửa ô "Cấu hình" nếu cần, rồi Run All. Kết quả ở `infinitetalk_out/<RUN_NAME>/` trong panel Output.

**Khi lỗi:** xem `infinitetalk_out/comfyui.log`. Hết VRAM: tăng `BLOCKS_TO_SWAP` (tối đa 40). Hết RAM: đổi `MAIN_MODEL` sang bản
`Q5_K_M`. Video ra màu đen hoặc nhiễu (tràn số fp16 trên T4): thử `VAE_PRECISION = "fp32"`. Nếu session bị dừng giữa chừng, các
khung đã xong vẫn nằm trong `frames/`.

## 1. Cấu hình

In [ ]:
CHARACTER_ID = "demo_host_02"  # one character = one customer, one shop
RUN_NAME = "preview"  # output folder name; use e.g. "train_v1" for the 3-5 minute run

# Inputs come from a private Kaggle dataset (Add Input); the first sorted match is used.
START_IMAGE_NAME = "start.png"
AUDIO_GLOB = "*.wav"
# Must be attached to the saved version: batch runs cannot attach datasets, kagglehub works interactively only.
INPUT_DATASET = "jintran0501/ai-live-infinitetalk-input"

# Same portrait size as the live clips (480x832) so the Ultralight avatar frames match the live scene (009).
WIDTH, HEIGHT = 480, 832
FPS = 25  # Ultralight with HuBERT needs 25 fps video
FRAME_WINDOW = 81  # frames per InfiniteTalk window
MOTION_FRAME = 9  # overlap frames carried into the next window

# Settings from kijai's InfiniteTalk I2V example (lightx2v distill LoRA: few steps, cfg 1).
STEPS = 6
CFG = 1.0
SHIFT = 11.0
SEED = 2
SCHEDULER = "dpm++_sde"
AUDIO_SCALE = 1.0

PROMPT = (
    "a young woman livestream host talks warmly to the camera, natural lip movement, gentle head motion, "
    "natural blinking, static camera, soft studio lighting"
)
NEGATIVE = (
    "bright tones, overexposed, static, blurred details, subtitles, style, works, paintings, images, overall gray, "
    "worst quality, low quality, JPEG compression residue, ugly, incomplete, extra fingers, poorly drawn hands, "
    "poorly drawn faces, deformed, disfigured, misshapen limbs, fused fingers, still picture, messy background, "
    "many people in the background, walking backwards, camera movement, zoom"
)

# T4 (sm75) has no bf16 or fp8 tensor cores: the model computes in fp16 with PyTorch SDPA attention.
# Q6_K (14 GB) instead of the example's Q8 (18 GB) so weights and offloaded blocks fit in Kaggle's ~29 GB RAM.
MAIN_MODEL = "wan2.1-i2v-14b-480p-Q6_K.gguf"
BASE_PRECISION = "fp16"
ATTENTION = "sdpa"
VAE_PRECISION = "fp16"
BLOCKS_TO_SWAP = 30  # of 40 transformer blocks kept in RAM; lower is faster if VRAM allows
TEXT_ENCODER_DEVICE = "gpu"  # falls back to "cpu" automatically if the T4 rejects bf16 matmul

RUN_TIMEOUT_S = 4 * 3600  # preview budget; raise to 11 h for the full run (Kaggle stops a session at 12 h)
EXTRA_SERVER_ARGS: list[str] = []

## 2. Môi trường, ComfyUI và WanVideoWrapper

In [ ]:
import json
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request
from datetime import UTC, datetime
from pathlib import Path

import torch  # preinstalled by Kaggle with the right CUDA build

ON_KAGGLE = Path("/kaggle").exists()
if not ON_KAGGLE:
    raise SystemExit("Notebook này chỉ chạy trên Kaggle (ADR-060), không chạy trên laptop.")

# Weights stay out of /kaggle/working: that folder is the saved output and is capped at 20 GB.
COMFY_DIR = Path("/tmp/ComfyUI")
OUT_ROOT = Path("/kaggle/working/infinitetalk_out")
OUT_DIR = OUT_ROOT / RUN_NAME
FRAMES_DIR = OUT_DIR / "frames"
FRAMES_DIR.mkdir(parents=True, exist_ok=True)

# Pinned so the API-format workflow below keeps matching the node inputs (checked against this commit).
WRAPPER_REPO = "https://github.com/kijai/ComfyUI-WanVideoWrapper"
WRAPPER_COMMIT = "a7439db30bed0c5b5fa437d77ae8a7bb220dfd00"
WRAPPER_DIR = COMFY_DIR / "custom_nodes" / "ComfyUI-WanVideoWrapper"
# The wrapper's own Wav2Vec2Model calls the encoder with output_hidden_states=True; transformers 5 no longer
# returns hidden states that way (None -> MultiTalkWav2VecEmbeds TypeError, run v4). 4.57.6 is the last 4.x.
TRANSFORMERS_PIN = "transformers==4.57.6"


def run(*cmd: str, cwd: Path | None = None) -> str:
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd[:4])} lỗi:\n{result.stderr[-3000:]}")
    return result.stdout.strip()


def pip_requirements(path: Path) -> list[str]:
    # Keep the platform's CUDA build of torch: reinstalling it from PyPI can break the GPU runtime.
    skip = {"torch", "torchvision", "torchaudio"}
    reqs = []
    for line in path.read_text().splitlines():
        line = line.strip()
        name = line.split(";")[0].split("[")[0]
        for sep in "<>=!~ ":
            name = name.split(sep)[0]
        if line and not line.startswith("#") and name.lower() not in skip:
            reqs.append(line)
    return reqs


if not COMFY_DIR.exists():
    run("git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI", str(COMFY_DIR))
COMFY_COMMIT = run("git", "rev-parse", "HEAD", cwd=COMFY_DIR)
if not WRAPPER_DIR.exists():
    WRAPPER_DIR.mkdir(parents=True)
    run("git", "init", "-q", cwd=WRAPPER_DIR)
    run("git", "fetch", "-q", "--depth", "1", WRAPPER_REPO, WRAPPER_COMMIT, cwd=WRAPPER_DIR)
    run("git", "checkout", "-q", "FETCH_HEAD", cwd=WRAPPER_DIR)

reqs = pip_requirements(COMFY_DIR / "requirements.txt") + pip_requirements(WRAPPER_DIR / "requirements.txt")
reqs = [r for r in reqs if not r.lower().startswith("transformers")] + [TRANSFORMERS_PIN]
run(sys.executable, "-m", "pip", "install", "-q", *reqs, "huggingface_hub")
TRANSFORMERS_VERSION = run(sys.executable, "-c", "import transformers; print(transformers.__version__)")

if shutil.which("ffmpeg") is None:
    run("apt-get", "install", "-y", "-qq", "ffmpeg")

GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
free_disk_gb = shutil.disk_usage("/tmp").free / 1e9
print(
    f"GPU: {GPU_NAME} · torch {torch.__version__} · transformers {TRANSFORMERS_VERSION}"
    f" · ComfyUI {COMFY_COMMIT[:10]} · wrapper {WRAPPER_COMMIT[:10]}"
)
print(f"Đĩa trống ở /tmp: {free_disk_gb:.0f} GB (cần khoảng 31 GB cho model)")
if GPU_NAME == "none":
    raise SystemExit("Chưa bật GPU. Kaggle: Accelerator = GPU T4 x2.")

## 3. Đầu vào: khung hình nhân vật và tiếng VieNeu

In [ ]:
INPUT_DIR = Path(globals().get("INPUT_DIR") or "/kaggle/input")


def find_inputs(root: Path) -> tuple[list[Path], list[Path]]:
    return sorted(root.rglob(START_IMAGE_NAME)), sorted(root.rglob(AUDIO_GLOB))


images, audios = find_inputs(INPUT_DIR)
if not images or not audios:
    # Interactive sessions only: a Save Version run refuses to attach a new dataset
    # ("New Datasets cannot be attached in non-interactive sessions").
    # A child process picks up an upgraded kagglehub without restarting the kernel.
    fetch = f"import kagglehub; print(kagglehub.dataset_download({INPUT_DATASET!r}))"
    try:
        run(sys.executable, "-m", "pip", "install", "-q", "-U", "kagglehub")
        INPUT_DIR = Path(run(sys.executable, "-c", fetch).splitlines()[-1])
        images, audios = find_inputs(INPUT_DIR)
        print(f"Lấy đầu vào từ dataset {INPUT_DATASET}: {INPUT_DIR}")
    except (RuntimeError, IndexError) as err:
        print(f"kagglehub không lấy được dataset {INPUT_DATASET}: {err}")
if not images or not audios:
    # Show what is mounted: a wrong file name is easier to spot than a missing input.
    mounted = sorted(str(p) for p in Path("/kaggle/input").rglob("*"))[:40]
    raise SystemExit(
        f"Không thấy {START_IMAGE_NAME} hoặc {AUDIO_GLOB} (đang có: {mounted}). "
        f"Add Input → chọn dataset {INPUT_DATASET}."
    )
START_IMAGE, AUDIO = images[0], audios[0]

probe = run(
    "ffprobe", "-v", "error", "-show_entries", "format=duration:stream=width,height", "-of", "json", str(START_IMAGE)
)
size = json.loads(probe)["streams"][0]
if (size["width"], size["height"]) != (WIDTH, HEIGHT):
    raise SystemExit(f"{START_IMAGE.name} là {size['width']}x{size['height']}, cần đúng {WIDTH}x{HEIGHT}.")
AUDIO_SECONDS = float(run("ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", str(AUDIO)))
NUM_FRAMES = round(AUDIO_SECONDS * FPS)
WINDOWS = 1 + max(0, -(-(NUM_FRAMES - FRAME_WINDOW) // (FRAME_WINDOW - MOTION_FRAME)))

(COMFY_DIR / "input").mkdir(exist_ok=True)
shutil.copy(START_IMAGE, COMFY_DIR / "input" / "start.png")
shutil.copy(AUDIO, COMFY_DIR / "input" / "speech.wav")
print(f"Ảnh: {START_IMAGE} · Tiếng: {AUDIO} ({AUDIO_SECONDS:.1f} s, {NUM_FRAMES} khung, khoảng {WINDOWS} cửa sổ)")

## 4. Tải model (chỉ trong máy Kaggle, không về laptop)

Khoảng 31 GB. License (đọc ngày 2026-11-17 trên Hugging Face): Wan2.1 GGUF, InfiniteTalk GGUF, lightx2v: Apache-2.0;
chinese-wav2vec2-base: MIT. TODO (ADR-060): đọc lại license từng bản trước khi khách trả tiền.

In [ ]:
from huggingface_hub import hf_hub_download

MODELS = {
    "wan_i2v": {"repo": "city96/Wan2.1-I2V-14B-480P-gguf", "file": MAIN_MODEL, "dir": "diffusion_models"},
    "infinitetalk": {
        "repo": "Kijai/WanVideo_comfy_GGUF",
        "file": "InfiniteTalk/Wan2_1-InfiniteTalk_Single_Q8.gguf",
        "dir": "diffusion_models",
    },
    "lightx2v_lora": {
        "repo": "Kijai/WanVideo_comfy",
        "file": "Lightx2v/lightx2v_I2V_14B_480p_cfg_step_distill_rank64_bf16.safetensors",
        "dir": "loras",
    },
    "vae": {"repo": "Kijai/WanVideo_comfy", "file": "Wan2_1_VAE_bf16.safetensors", "dir": "vae"},
    "text_encoder": {"repo": "Kijai/WanVideo_comfy", "file": "umt5-xxl-enc-bf16.safetensors", "dir": "text_encoders"},
    "clip_vision": {
        "repo": "Comfy-Org/Wan_2.1_ComfyUI_repackaged",
        "file": "split_files/clip_vision/clip_vision_h.safetensors",
        "dir": "clip_vision",
    },
    "wav2vec": {
        "repo": "Kijai/wav2vec2_safetensors",
        "file": "wav2vec2-chinese-base_fp16.safetensors",
        "dir": "wav2vec2",
    },
}

for key, model in MODELS.items():
    started = time.monotonic()
    cached = Path(hf_hub_download(model["repo"], model["file"]))
    parts = cached.parts
    model["revision"] = parts[parts.index("snapshots") + 1] if "snapshots" in parts else "unknown"
    target = COMFY_DIR / "models" / model["dir"] / cached.name
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        target.symlink_to(cached)
    print(f"{key}: {cached.name} @ {model['revision'][:10]} ({time.monotonic() - started:.0f} s)")

## 5. Bật ComfyUI (chỉ nghe trong máy, không mở giao diện web)

In [ ]:
SERVER = "http://127.0.0.1:8188"
LOG_PATH = OUT_ROOT / "comfyui.log"
# Stays open while the server runs; the notebook process owns it.
server_log = open(LOG_PATH, "w")
server = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", "--preview-method", "none"]
    + EXTRA_SERVER_ARGS,
    cwd=COMFY_DIR,
    stdout=server_log,
    stderr=subprocess.STDOUT,
)


def api(path: str, payload: dict | None = None) -> dict:
    data = None if payload is None else json.dumps(payload).encode()
    request = urllib.request.Request(SERVER + path, data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=30) as response:
        body = response.read()
    return json.loads(body) if body else {}


def log_tail(chars: int = 3000) -> str:
    return LOG_PATH.read_text(errors="replace")[-chars:]


for _ in range(180):
    if server.poll() is not None:
        raise RuntimeError("ComfyUI dừng khi khởi động:\n" + log_tail())
    try:
        api("/system_stats")
        break
    except OSError:
        time.sleep(2)
else:
    raise RuntimeError("ComfyUI không lên sau 6 phút:\n" + log_tail())

missing = [n for n in ("WanVideoSampler", "MultiTalkWav2VecEmbeds") if n not in api("/object_info")]
if missing:
    raise RuntimeError(f"WanVideoWrapper chưa nạp được ({missing}):\n" + log_tail())
print("ComfyUI và WanVideoWrapper đã chạy.")


def run_workflow(workflow: dict, timeout_s: float, on_wait=None) -> dict:
    """Queue one API-format workflow and wait for its outputs."""
    try:
        prompt_id = api("/prompt", {"prompt": workflow})["prompt_id"]
    except urllib.error.HTTPError as err:
        # ComfyUI explains a rejected workflow (missing node, wrong input) in the body.
        raise RuntimeError(err.read().decode(errors="replace")[:3000]) from err
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        entry = api(f"/history/{prompt_id}").get(prompt_id)
        if entry is not None:
            if entry.get("status", {}).get("status_str") == "error":
                raise RuntimeError(json.dumps(entry["status"], ensure_ascii=False)[:3000])
            return entry["outputs"]
        if server.poll() is not None:
            raise RuntimeError("ComfyUI đã dừng giữa chừng:\n" + log_tail())
        if on_wait is not None:
            on_wait()
        time.sleep(10)
    raise TimeoutError(f"Quá {timeout_s:.0f} s mà workflow chưa xong")

## 6. Tạo video (InfiniteTalk)

Mỗi cửa sổ lưu khung PNG vào `frames/` ngay khi xong, nên session bị dừng thì phần đã làm vẫn còn. Ô này in tiến độ mỗi 5 phút.

In [ ]:
def name_of(key: str) -> str:
    return Path(MODELS[key]["file"]).name


def talk_workflow(text_device: str) -> dict:
    return {
        "1": {"class_type": "MultiTalkModelLoader", "inputs": {"model": name_of("infinitetalk")}},
        "2": {
            "class_type": "WanVideoBlockSwap",
            "inputs": {
                "blocks_to_swap": BLOCKS_TO_SWAP,
                "offload_img_emb": False,
                "offload_txt_emb": False,
                "use_non_blocking": True,
                "vace_blocks_to_swap": 0,
                "prefetch_blocks": 1,
                "block_swap_debug": False,
            },
        },
        # GGUF weights cannot merge LoRAs, so the wrapper applies it on the fly.
        "3": {
            "class_type": "WanVideoLoraSelect",
            "inputs": {"lora": name_of("lightx2v_lora"), "strength": 1.0, "low_mem_load": False, "merge_loras": False},
        },
        "4": {
            "class_type": "WanVideoModelLoader",
            "inputs": {
                "model": name_of("wan_i2v"),
                "base_precision": BASE_PRECISION,
                "quantization": "disabled",
                "load_device": "offload_device",
                "attention_mode": ATTENTION,
                "block_swap_args": ["2", 0],
                "lora": ["3", 0],
                "multitalk_model": ["1", 0],
            },
        },
        "5": {"class_type": "WanVideoVAELoader", "inputs": {"model_name": name_of("vae"), "precision": VAE_PRECISION}},
        "6": {"class_type": "CLIPVisionLoader", "inputs": {"clip_name": name_of("clip_vision")}},
        "7": {"class_type": "LoadImage", "inputs": {"image": "start.png"}},
        "8": {
            "class_type": "WanVideoClipVisionEncode",
            "inputs": {
                "clip_vision": ["6", 0],
                "image_1": ["7", 0],
                "strength_1": 1.0,
                "strength_2": 1.0,
                "crop": "center",
                "combine_embeds": "average",
                "force_offload": True,
            },
        },
        "9": {
            "class_type": "WanVideoImageToVideoMultiTalk",
            "inputs": {
                "vae": ["5", 0],
                "width": WIDTH,
                "height": HEIGHT,
                "frame_window_size": FRAME_WINDOW,
                "motion_frame": MOTION_FRAME,
                "force_offload": True,
                "colormatch": "disabled",
                "start_image": ["7", 0],
                "tiled_vae": False,
                "clip_embeds": ["8", 0],
                "mode": "infinitetalk",
                # Frames go to disk window by window instead of one big tensor at the end.
                "output_path": str(FRAMES_DIR),
            },
        },
        "10": {
            "class_type": "Wav2VecModelLoader",
            "inputs": {"model": name_of("wav2vec"), "base_precision": "fp16", "load_device": "main_device"},
        },
        "11": {"class_type": "LoadAudio", "inputs": {"audio": "speech.wav"}},
        "12": {
            "class_type": "MultiTalkWav2VecEmbeds",
            "inputs": {
                "wav2vec_model": ["10", 0],
                "audio_1": ["11", 0],
                "normalize_loudness": True,
                "num_frames": NUM_FRAMES,
                "fps": float(FPS),
                "audio_scale": AUDIO_SCALE,
                "audio_cfg_scale": 1.0,
                "multi_audio_type": "para",
            },
        },
        "13": {
            "class_type": "WanVideoTextEncodeCached",
            "inputs": {
                "model_name": name_of("text_encoder"),
                "precision": "bf16",
                "positive_prompt": PROMPT,
                "negative_prompt": NEGATIVE,
                "quantization": "disabled",
                "use_disk_cache": True,
                "device": text_device,
            },
        },
        "14": {
            "class_type": "WanVideoSampler",
            "inputs": {
                "model": ["4", 0],
                "image_embeds": ["9", 0],
                "text_embeds": ["13", 0],
                "multitalk_embeds": ["12", 0],
                "steps": STEPS,
                "cfg": CFG,
                "shift": SHIFT,
                "seed": SEED,
                "force_offload": True,
                "scheduler": SCHEDULER,
                "riflex_freq_index": 0,
                "denoise_strength": 1.0,
                "batched_cfg": False,
                "rope_function": "comfy",
                "start_step": 0,
                "end_step": -1,
                "add_noise_to_samples": False,
            },
        },
        # With output_path set the sampler returns a dummy frame; this output node only makes ComfyUI run the graph.
        "15": {"class_type": "WanVideoPassImagesFromSamples", "inputs": {"samples": ["14", 0]}},
        "16": {"class_type": "PreviewImage", "inputs": {"images": ["15", 0]}},
    }


def frame_files() -> list[Path]:
    return sorted(FRAMES_DIR.glob("frame_*.png"))


last_report = [time.monotonic()]


def report_progress() -> None:
    if time.monotonic() - last_report[0] < 300:
        return
    last_report[0] = time.monotonic()
    done = len(frame_files())
    elapsed = time.monotonic() - started
    print(f"[{elapsed / 60:.0f} phút] {done}/{NUM_FRAMES} khung", flush=True)


started = time.monotonic()
text_device = TEXT_ENCODER_DEVICE
try:
    run_workflow(talk_workflow(text_device), RUN_TIMEOUT_S, report_progress)
except RuntimeError as err:
    # Some T4 runtimes reject bf16 matmul; the text encoder is the only bf16 step, so retry it on the CPU.
    if text_device != "gpu" or "WanVideoTextEncodeCached" not in str(err):
        raise
    print("Bộ mã hóa chữ lỗi trên GPU, thử lại trên CPU:", str(err)[:500])
    text_device = "cpu"
    run_workflow(talk_workflow(text_device), RUN_TIMEOUT_S, report_progress)
GEN_SECONDS = round(time.monotonic() - started, 1)
frames = frame_files()
print(f"Xong {len(frames)} khung trong {GEN_SECONDS / 60:.1f} phút ({len(frames) / GEN_SECONDS:.3f} khung/giây)")

## 7. Ghép video, đo tốc độ từng cửa sổ, ghi `run.json`

In [ ]:
from IPython.display import Video, display

VIDEO = OUT_DIR / f"{CHARACTER_ID}_{RUN_NAME}.mp4"
run(
    "ffmpeg", "-y", "-loglevel", "error", "-framerate", str(FPS), "-i", str(FRAMES_DIR / "frame_%05d.png"),
    "-i", str(AUDIO), "-map", "0:v", "-map", "1:a", "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "14",
    "-c:a", "aac", "-b:a", "192k", "-shortest", str(VIDEO),
)  # fmt: skip

# Each window writes its frames in one burst, so gaps between PNG write times mark window boundaries.
stamps = [f.stat().st_mtime for f in frames]
windows, burst_start = [], 0
for i in range(1, len(stamps) + 1):
    if i == len(stamps) or stamps[i] - stamps[i - 1] > 20:
        windows.append({"frames": i - burst_start, "finished_at": stamps[i - 1]})
        burst_start = i
gen_start = time.time() - (time.monotonic() - started)
previous = gen_start
for window in windows:
    window["seconds"] = round(window["finished_at"] - previous, 1)
    previous = window.pop("finished_at")
steady = [w["seconds"] for w in windows[1:]] or [w["seconds"] for w in windows]
print("Thời gian từng cửa sổ (giây):", [w["seconds"] for w in windows])
print(f"Cửa sổ thường: {sum(steady) / len(steady) / 60:.1f} phút cho {FRAME_WINDOW - MOTION_FRAME} khung mới")

record = {
    "character_id": CHARACTER_ID,
    "run_name": RUN_NAME,
    "created_at": datetime.now(UTC).isoformat(timespec="seconds"),
    "purpose": "Ultralight training data only, never streamed live (ADR-060)",
    "gpu": GPU_NAME,
    "torch": torch.__version__,
    "transformers": TRANSFORMERS_VERSION,
    "comfyui_commit": COMFY_COMMIT,
    "wrapper_commit": WRAPPER_COMMIT,
    "models": MODELS,
    "inputs": {"start_image": START_IMAGE.name, "audio": AUDIO.name, "audio_seconds": AUDIO_SECONDS},
    "settings": {
        "size": [WIDTH, HEIGHT],
        "fps": FPS,
        "frame_window": FRAME_WINDOW,
        "motion_frame": MOTION_FRAME,
        "steps": STEPS,
        "cfg": CFG,
        "shift": SHIFT,
        "seed": SEED,
        "scheduler": SCHEDULER,
        "audio_scale": AUDIO_SCALE,
        "base_precision": BASE_PRECISION,
        "attention": ATTENTION,
        "vae_precision": VAE_PRECISION,
        "blocks_to_swap": BLOCKS_TO_SWAP,
        "text_encoder_device": text_device,
        "prompt": PROMPT,
        "negative": NEGATIVE,
    },  # fmt: skip
    "timing": {
        "generation_seconds": GEN_SECONDS,
        "frames": len(frames),
        "frames_per_second": round(len(frames) / GEN_SECONDS, 4),
        "windows": windows,
    },
    "video": VIDEO.name,
}
(OUT_DIR / "run.json").write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
shutil.copy(LOG_PATH, OUT_DIR / "comfyui.log")
# The mp4 at crf 14 keeps what Ultralight needs; thousands of PNGs would crowd the 20 GB output folder.
shutil.rmtree(FRAMES_DIR)
print(f"Đã lưu {VIDEO} và run.json. Tải về assets/avatars/{CHARACTER_ID}/training/ (gitignore), không commit.")
display(Video(str(VIDEO), embed=True, width=240))